<center><img src="../Picture Data/logo.png" alt="Header" style="width: 800px;"/></center>

@Copyright (C): 2010-2019, Shenzhen Yahboom Tech  
@Author: Malloy.Yuan  
@Date: 2019-07-17 10:10:02  
@LastEditors: Malloy.Yuan  
@LastEditTime: 2019-09-17 17:54:19  

# Handle Control Car
In this example, we will remotely control Jetbot using a gamepad controller connected to the web browser machine.

### Create Handle controller
First, we need to create an instance of the 'Controller' widget, which we will use to drive our Jetbot. The "Controller" widget accepts an "index" parameter that specifies the number of controllers.

1. Open [http://html5gamepad.com](http://html5gamepad.com).  
2. If you press the button of the handle you are using
3. Record the corresponding index number that pops up when you press the button

Next, we will use this index to create and display the controller.

In [ ]:
import time
import ipywidgets as widgets
from IPython.display import display
from jetbot import Robot, Camera, bgr8_to_jpeg

robot = Robot()
camera = Camera.instance(width=224, height=224)

print("Robot and camera initialized")

In [ ]:
type(camera.value), camera.value.shape, camera.value.dtype

Import some module

In [ ]:
import traitlets

image = widgets.Image(format='jpeg', width=224, height=224)

try:
    camera_link.unlink()
except Exception:
    pass

camera_link = traitlets.dlink(
    (camera, 'value'),
    (image, 'value'),
    transform=bgr8_to_jpeg
)

display(image)

Create and open a display camera initialization instance

In [ ]:
import threading
from servoserial import ServoSerial

CONTROLLER_INDEX = 0

controller = widgets.Controller(index=CONTROLLER_INDEX)
display(controller)

status = widgets.HTML()
stop_button = widgets.Button(description="STOP", button_style="danger")
display(stop_button, status)

# Yahboom Handle control.ipynb 기준
DRIVE_X_AXIS = 0      # left stick 좌우
DRIVE_Y_AXIS = 1      # left stick 전후
CAM_UD_AXIS = 2       # camera up/down
CAM_LR_AXIS = 5       # camera left/right
CAM_RESET_BUTTON = 8  # SELECT reset

DEADZONE = 0.10
SPEED_LIMIT = 0.5
SERVO_STEP = 15

PAN_CENTER = 2048
TILT_MIN = 800
TILT_CENTER = 1050
TILT_MAX = 1700

servo_device = ServoSerial()

leftrightpulse = PAN_CENTER
updownpulse = TILT_CENTER

control_stop = threading.Event()
control_thread = None

def clamp(value, low=-1.0, high=1.0):
    return max(low, min(high, value))

def apply_deadzone(value):
    return 0.0 if abs(value) < DEADZONE else value

def axis_value(axis_index):
    if axis_index >= len(controller.axes):
        return 0.0
    return float(controller.axes[axis_index].value or 0.0)

def button_value(button_index):
    if button_index >= len(controller.buttons):
        return False
    return bool(controller.buttons[button_index].value)

def cam_up():
    global updownpulse
    updownpulse = min(TILT_MAX, updownpulse + SERVO_STEP)
    servo_device.Servo_serial_control(2, updownpulse)

def cam_down():
    global updownpulse
    updownpulse = max(TILT_MIN, updownpulse - SERVO_STEP)
    servo_device.Servo_serial_control(2, updownpulse)

def cam_left():
    global leftrightpulse
    leftrightpulse = min(3600, leftrightpulse + SERVO_STEP)
    servo_device.Servo_serial_control(1, leftrightpulse)

def cam_right():
    global leftrightpulse
    leftrightpulse = max(600, leftrightpulse - SERVO_STEP)
    servo_device.Servo_serial_control(1, leftrightpulse)

def cam_center():
    global leftrightpulse, updownpulse
    leftrightpulse = PAN_CENTER
    updownpulse = TILT_CENTER
    servo_device.Servo_serial_control(1, leftrightpulse)
    time.sleep(0.1)
    servo_device.Servo_serial_control(2, updownpulse)

def drive_from_left_stick():
    x = apply_deadzone(axis_value(DRIVE_X_AXIS))
    y = apply_deadzone(axis_value(DRIVE_Y_AXIS))

    # gamepad는 보통 앞으로 밀면 y가 음수라서 -y 사용
    throttle = -y
    turn = x

    left = clamp((throttle + turn) * SPEED_LIMIT)
    right = clamp((throttle - turn) * SPEED_LIMIT)

    if left == 0.0 and right == 0.0:
        robot.stop()
    else:
        robot.set_motors(left, right)

    return left, right

def control_loop():
    cam_ud_count = 0
    cam_lr_count = 0
    reset_count = 0

    while not control_stop.is_set():
        left, right = drive_from_left_stick()

        cam_ud = axis_value(CAM_UD_AXIS)
        cam_lr = axis_value(CAM_LR_AXIS)

        if cam_ud == 1:
            cam_ud_count += 1
            if cam_ud_count >= 3:
                cam_down()
                cam_ud_count = 0
        elif cam_ud == -1:
            cam_ud_count += 1
            if cam_ud_count >= 3:
                cam_up()
                cam_ud_count = 0
        else:
            cam_ud_count = 0

        if cam_lr == 1:
            cam_lr_count += 1
            if cam_lr_count >= 3:
                cam_right()
                cam_lr_count = 0
        elif cam_lr == -1:
            cam_lr_count += 1
            if cam_lr_count >= 3:
                cam_left()
                cam_lr_count = 0
        else:
            cam_lr_count = 0

        if button_value(CAM_RESET_BUTTON):
            reset_count += 1
            if reset_count >= 3:
                cam_center()
                reset_count = 0
        else:
            reset_count = 0

        status.value = f"left={left:.2f}, right={right:.2f}, pan={leftrightpulse}, tilt={updownpulse}"
        time.sleep(0.02)

def start_control():
    global control_thread
    control_stop.clear()
    if control_thread is None or not control_thread.is_alive():
        control_thread = threading.Thread(target=control_loop, daemon=True)
        control_thread.start()

def stop(change=None):
    control_stop.set()
    robot.stop()
    status.value = "STOPPED"

stop_button.on_click(stop)

cam_center()
start_control()

print("Controller control loop started")

Add heartbeat connection

In [ ]:
try:
    control_stop.set()
except NameError:
    pass

try:
    robot.stop()
except NameError:
    pass

try:
    camera.unobserve(update_image, names='value')
except Exception as e:
    print("camera unobserve skipped:", e)

try:
    camera.stop()
except Exception as e:
    print("camera stop skipped:", e)

print("Stopped safely")

Create a method to actively stop the process

In [ ]:
robot.stop()

Method for creating up, down, left, and right movements of a PTZ camera separately

In [ ]:
camera.unobserve(update_image, names='value')
robot.stop()
print("Stopped camera observer and robot")

In [ ]:
camera.stop()

### Initialize the position of the PTZ camera
Run the following cell code to initialize the PTZ project to the initial location

In [ ]:
camservoInitFunction()

### Load the Robot class
This class allows us to easily control the JetBot motor

In [ ]:
robot = Robot()

### Create onboard breathing light method

In [ ]:
def BLN_Onboard():
    global i , k
    i = k = 0
    while True:
        if k == 0:
            robot.set_bln(i)
            i += 0.01
            if( i >= 1 ):
                k = 1
            time.sleep(0.005)
        elif k == 1:
            robot.set_bln(i)
            i -=0.01
            if i <= 0 :
                k = 0
            time.sleep(0.005)

In [ ]:
# import time
# robot.forward(1)
# time.sleep(0.5)
# robot.stop()

Turn on the onboard breathing light independent process by running the cell code below

In [ ]:
thread1 = threading.Thread(target=BLN_Onboard)
# thread1.setDaemon(True)
thread1.start()

<center><img src="../Picture Data/Handle.png" alt="Header" style="width: 400px;"/></center>

### Create a handle to control the movement of Jetbot in real time
### Please use the ANOLOG button in the middle of the handle to switch to the simulation mode before using.

Program features:
          1. Left rocker control Jetbot movement, right rocker control the servo movement
          2. Press the SELECT button to reset the PTZ angle
          3. Press the L side button No.1 to control the PTZ rise, press the L side button No.2 to control the gimbal decline.

In the default code, the Yahboom handle is used by default. For other handles, please refer to the key table to change the value.
     
     If you are using Yahboom accessory, please use the code section ---1
     If you are using the Xbox360's handle, please use the code section ---2

In [ ]:
def jetbot_motion():
    count1 = count2 = count3 = count4 =  count5 = 0
    while 1:
        #Robot car Left and right DC motor
        #Handle control code---1(Jetbot Yahboom handle)
        #Yahboom Rocker reset value is 0.0039,
        #In order to prevent the motor from being abnormal, the following code is added to operate.
        if controller.axes[1].value <= 0.1:
            if (controller.axes[0].value <= 0.1 and controller.axes[0].value >= -0.1 
                and controller.axes[1].value <= 0.1 and controller.axes[1].value >= -0.1):
                robot.stop()
            else:
                robot.set_motors(-controller.axes[1].value + controller.axes[0].value, -controller.axes[1].value - controller.axes[0].value)
            
            time.sleep(0.01)
        else:
            robot.set_motors(-controller.axes[1].value - controller.axes[0].value, -controller.axes[1].value + controller.axes[0].value)
            time.sleep(0.01)
          #Handle control code---2(Xbox360手柄)
#         if controller.axes[1].value <= 0:
#             robot.set_motors(-controller.axes[1].value + controller.axes[0].value, -controller.axes[1].value - controller.axes[0].value)
#             time.sleep(0.01)
#         else:
#             robot.set_motors(-controller.axes[1].value - controller.axes[0].value, -controller.axes[1].value + controller.axes[0].value)
#             time.sleep(0.01)

        #Servo control camera up and down 
        if controller.axes[2].value == 1:
            count1  += 1
            if count1 >= 3:
                camDownFunction()
                count1 = 0
        elif controller.axes[2].value == -1:
            count1  += 1
            if count1 >= 3:
                camUpFunction()
                count1 = 0
        else:
            count1 = 0
        #Servo control camera left and right
        if controller.axes[5].value == 1:
            count2  += 1
            if count2 >= 3:
                camRightFunction()
                count2 = 0
        elif controller.axes[5].value == -1:
            count2  += 1
            if count2 >= 3:
                camLeftFunction()
                count2 = 0
        else:
            count2 = 0
        #Servo control camera up down, left and right is reset
        if controller.buttons[8].value == True:
            count3 += 1
            if count3 >= 3:
                camservoInitFunction()
                count3 = 0
        else:
            count3 = 0
        
        #Servo control servo rise and decline
        if controller.buttons[6].value == True:
            count4 += 1
            if count4 >= 3:
                robot.set_vertical_motors(1)
                count4 = 0
        elif controller.buttons[4].value == True:
            count4 += 1
            if count4 >= 3:
                robot.set_vertical_motors(-1)
                count4 = 0
        else:
            robot.set_vertical_motors(0)
            count4 = 0

Control the independent process of Jetbot motion in real time by running the cell code below to open the handle

In [ ]:
thread2 = threading.Thread(target=jetbot_motion)
thread2.setDaemon(True)
thread2.start()

Ending onboard breathing RGB process

In [ ]:
stop_thread(thread1)

Ending Jetbot movement process

In [ ]:
stop_thread(thread2)
robot.stop()

In [ ]:
stop_thread(thread2)